# TP558 — Demonstração didática de constantes de tempo adaptativas

**Artigo-base:** Hasani et al. (2020), *Liquid Time-constant Networks* (arXiv:2006.04439).

Este notebook **não reproduz o treinamento nem os resultados numéricos do artigo**. É uma experiência didática inspirada no mecanismo de relaxação variável de uma LTC, usando um sistema escalar simplificado:

$$\dot x(t) = -[1/\tau+f(I(t))]x(t)+f(I(t))A.$$

Comparamos $f$ constante e $f$ dependente da entrada, usando Euler explícito. Execute as células na ordem.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

dt, T, tau, A = 0.005, 12.0, 2.0, 1.0
t = np.arange(0, T + dt, dt)
I = np.where((t >= 2) & (t < 6), 1., np.where(t >= 8, 0.6, 0.))

def integrate(f_values):
    x = np.zeros_like(t)
    for k in range(len(t)-1):
        dx = -(1/tau + f_values[k])*x[k] + f_values[k]*A
        x[k+1] = x[k] + dt*dx
    return x

f_fixed = np.full_like(t, 0.65)
f_adaptive = 0.10 + 2.0*I
x_fixed, x_adaptive = integrate(f_fixed), integrate(f_adaptive)
tau_fixed = 1/(1/tau + f_fixed)
tau_adaptive = 1/(1/tau + f_adaptive)
print('Intervalo de τ efetivo:')
print('Fixo:', round(tau_fixed.min(),3))
print('Adaptativo:',round(tau_adaptive.min(),3), 'a', round(tau_adaptive.max(),3))

In [ ]:
fig, axes=plt.subplots(3,1,figsize=(11,8),sharex=True)
axes[0].plot(t,I,label='Entrada I(t)');axes[0].set_ylabel('Entrada');axes[0].legend()
axes[1].plot(t,x_fixed,label='Relaxação fixa')
axes[1].plot(t,x_adaptive,label='Relaxação adaptativa')
axes[1].set_ylabel('Estado x(t)');axes[1].legend()
axes[2].plot(t,tau_fixed,label='τ efetivo fixo')
axes[2].plot(t,tau_adaptive,label='τ efetivo adaptativo')
axes[2].set_ylabel('τ efetivo');axes[2].set_xlabel('Tempo (unidades arbitrárias)');axes[2].legend()
for ax in axes:ax.grid(True,alpha=.25)
plt.tight_layout();plt.show()

## Interpretação e limitações

1. Quando a entrada sobe, $f(I)$ aumenta, reduzindo $\tau_{\mathrm{efetivo}}=1/(1/\tau+f(I))$.
2. Isso modifica a velocidade de evolução do estado. Observe também que **o ponto de equilíbrio muda**: $x^*=f A/(1/\tau+f)$.
3. O experimento não contém pesos treinados, múltiplos neurônios, solver fused ou benchmarks do paper. Portanto, **não é evidência experimental de superioridade de LTC**.

**Perguntas para discussão:** O que acontece ao aumentar $dt$? Como diferenciar alteração da velocidade e alteração do equilíbrio? Quais métricas seriam relevantes em um cenário real?